In [1]:

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from uszipcode import SearchEngine

#import tensorflow_datasets as tfds

In [2]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [3]:
# Sourced directly from TruckerPath
park_data_1 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_1 - Copy.csv")
park_data_2 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_2 - Copy.csv")
park_data_3 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_3 - Copy.csv")
park_data = pd.concat([park_data_1, park_data_2, park_data_3], ignore_index=True)

In [4]:
truck_stop = pd.read_excel("output_excel\Model_Stops_V3.xlsx")

In [5]:
avail_park = park_data[park_data["pin id"].isin(truck_stop["pin id"].unique())].copy()

In [6]:
avail_park["time(utc)"] = pd.to_datetime(avail_park["time(utc)"])

In [7]:
avail_park["day_of_week"] = avail_park["time(utc)"].dt.dayofweek
avail_park["day_name"] = avail_park["time(utc)"].dt.day_name()
avail_park["hour"] = avail_park["time(utc)"].dt.hour
avail_park["month"] = avail_park["time(utc)"].dt.month
avail_park["date"] = avail_park["time(utc)"].dt.day

In [8]:
t_df = avail_park[["pinname", "pinlat", "pinlon"]].drop_duplicates()

In [9]:


search = SearchEngine()


def get_zip(row):
    r = search.by_coordinates(row['pinlat'], row['pinlon'], radius=30, returns=1)
    return r[0].zipcode if r else None


t_df['ZIP5'] = t_df.apply(get_zip, axis=1)

t_df['ZIP3'] = t_df['ZIP5'].str[:3]

In [10]:
avail_park = pd.merge(avail_park, t_df, on=["pinname", "pinlat", "pinlon"], how='left')

In [11]:
traffic_df = pd.read_csv("ZIP3_agg_data.csv", dtype={"ZIP3": "str"})

In [12]:
traffic_df = traffic_df.groupby(["ZIP3", "day_of_week", "hours"]).agg({"avg_traffic": "mean"}).reset_index()
traffic_df["hours"] = traffic_df["hours"].str.replace("hour_", "")
traffic_df["hour"] = traffic_df["hours"].astype("int")
avail_park = pd.merge(avail_park, traffic_df, on=["ZIP3", "day_of_week", "hour"], how="left")

In [13]:
avail_park

,pinname,parking status,time(utc),pinlat,pinlon,pin id,object,city,route_num,day_of_week,day_name,hour,month,date,ZIP5,ZIP3,hours,avg_traffic
0,Rest Area WB,Lots,2024-11-22 22:48:15,30.612343,-86.978559,005b00cc243fb4f39296fa0f16a21482,Rest Area,Milton,10,4,Friday,22,11,22,32583,325,22,233.526126
1,Rest Area WB,Lots,2024-11-22 18:51:59,30.612343,-86.978559,005b00cc243fb4f39296fa0f16a21482,Rest Area,Milton,10,4,Friday,18,11,22,32583,325,18,747.844565
2,Rest Area WB,Lots,2024-11-22 18:46:59,30.612343,-86.978559,005b00cc243fb4f39296fa0f16a21482,Rest Area,Milton,10,4,Friday,18,11,22,32583,325,18,747.844565
3,Rest Area WB,Lots,2024-11-22 15:30:00,30.612343,-86.978559,005b00cc243fb4f39296fa0f16a21482,Rest Area,Milton,10,4,Friday,15,11,22,32583,325,15,1064.738264
4,Rest Area WB,Lots,2024-11-22 15:11:00,30.612343,-86.978559,005b00cc243fb4f39296fa0f16a21482,Rest Area,Milton,10,4,Friday,15,11,22,32583,325,15,1064.738264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1559599,ONE9 Dealer (One9 Fuel Network) #1365,Lots,2025-11-05 21:16:49,32.744745,-85.279124,928e734545f99a375312f4a20606f560,One9,Cusseta,85,2,Wednesday,21,11,5,36852,368,21,218.464562
1559600,ONE9 Dealer (One9 Fuel Network) #1365,Lots,2025-11-05 16:37:19,32.744745,-85.279124,928e734545f99a375312f4a20606f560,One9,Cusseta,85,2,Wednesday,16,11,5,36852,368,16,890.379613
1559601,ONE9 Dealer (One9 Fuel Network) #1365,Some,2025-11-05 01:41:36,32.744745,-85.279124,928e734545f99a375312f4a20606f560,One9,Cusseta,85,2,Wednesday,1,11,5,36852,368,01,55.940162
1559602,ONE9 Dealer (One9 Fuel Network) #1365,Lots,2025-09-13 20:16:23,32.744745,-85.279124,928e734545f99a375312f4a20606f560,One9,Cusseta,85,5,Saturday,20,9,13,36852,368,20,398.526313


In [29]:
avail_park = avail_park[avail_park["parking status"] != 'Paid'].copy()
reg_df = avail_park[avail_park['pin id'] == '0583a5874b6f5b0b792d133c942ef95c'].copy()
# reg_df = avail_park.copy()

In [38]:
# --------------------------------------------------------------
# RUN TESTS FOR ALL LOCATIONS
# --------------------------------------------------------------

results = []  # will store metrics for every pin id

for pin in avail_park["pin id"].unique():

    reg_df = avail_park[avail_park["pin id"] == pin].copy()

    # Skip small datasets: too few observations = noisy statistics
    if len(reg_df) < 50:
        continue

    # ---- Chi-square: DAY OF WEEK ----
    try:
        ct = pd.crosstab(reg_df["day_of_week"], reg_df["parking status"])
        chi2_dow, p_dow, _, _ = chi2_contingency(ct)
        n = ct.to_numpy().sum()
        phi2 = chi2_dow / n
        r, k = ct.shape
        cramer_dow = np.sqrt(phi2 / min(r - 1, k - 1))
    except:
        cramer_dow = np.nan

    # ---- Chi-square: HOUR ----
    try:
        ct = pd.crosstab(reg_df["hour"], reg_df["parking status"])
        chi2_hour, p_hour, _, _ = chi2_contingency(ct)
        n = ct.to_numpy().sum()
        phi2 = chi2_hour / n
        r, k = ct.shape
        cramer_hour = np.sqrt(phi2 / min(r - 1, k - 1))
    except:
        cramer_hour = np.nan

    # ---- Mutual Information ----
    try:
        le_y = LabelEncoder()
        y_encoded = le_y.fit_transform(reg_df["parking status"])

        X_enc = reg_df[["hour", "day_of_week", "month", "date", "avg_traffic"]].copy()
        for col in X_enc.columns:
            le = LabelEncoder()
            X_enc[col] = le.fit_transform(X_enc[col])

        mi_vals = mutual_info_classif(X_enc, y_encoded, discrete_features=True)
        mi_hour = mi_vals[0]
        mi_dow = mi_vals[1]
        mi_month = mi_vals[2]
        mi_date = mi_vals[3]
        mi_traffic = mi_vals[4]

    except:
        mi_hour = mi_dow = mi_month = mi_date = mi_traffic = np.nan

    # ---- Eta Squared ----
    try:
        eta2 = eta_squared(reg_df, "avg_traffic", "parking status")
    except:
        eta2 = np.nan

    # ---- Append results ----
    results.append({
        "pin_id": pin,
        "n_obs": len(reg_df),
        "p_dow": p_dow,
        "p_hour": p_hour,
        "cramer_hour": cramer_hour,
        "cramer_dow": cramer_dow,
        "mi_hour": mi_hour,
        "mi_dow": mi_dow,
        "mi_month": mi_month,
        "mi_date": mi_date,
        "mi_traffic": mi_traffic,
        "eta2_traffic": eta2
    })

# Convert results into dataframe
out_df = pd.DataFrame(results)
# print(out_df.head())

# Save for analysis
out_df.to_csv("location_feature_relationships.csv", index=False)


C:\Users\bhavy\miniconda3\envs\truck_cap_2\lib\site-packages\sklearn\metrics\cluster\_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
C:\Users\bhavy\miniconda3\envs\truck_cap_2\lib\site-packages\sklearn\metrics\cluster\_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
C:\Users\bhavy\miniconda3\envs\truck_cap_2\lib\site-packages\sklearn\metrics\cluster\_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
C:\Users\bhavy\miniconda3\envs\truck_cap_2\lib\site-packages\sklearn\metrics\cl

In [30]:
reg_df

,pinname,parking status,time(utc),pinlat,pinlon,pin id,object,city,route_num,day_of_week,day_name,hour,month,date,ZIP5,ZIP3,hours,avg_traffic
1300573,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Full,2025-08-21 16:24:21,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,3,Thursday,16,8,21,32277,322,16,1727.378369
1300574,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Full,2025-08-21 15:45:21,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,3,Thursday,15,8,21,32277,322,15,1667.860377
1300575,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Some,2025-08-21 14:44:23,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,3,Thursday,14,8,21,32277,322,14,1475.526492
1300576,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Some,2025-08-21 13:59:23,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,3,Thursday,13,8,21,32277,322,13,1337.626758
1300577,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Lots,2025-08-21 03:07:18,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,3,Thursday,3,8,21,32277,322,03,130.999096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301790,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Full,2025-09-08 03:28:41,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,0,Monday,3,9,8,32277,322,NaN,NaN
1301791,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Some,2025-09-07 19:43:42,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,6,Sunday,19,9,7,32277,322,19,1072.551889
1301792,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Some,2025-09-07 18:50:42,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,6,Sunday,18,9,7,32277,322,18,1378.463870
1301793,Mr. Fuel Travel Center (One9 Fuel Network) #1149,Full,2025-09-07 04:57:04,30.412928,-81.571693,0583a5874b6f5b0b792d133c942ef95c,One9,Jacksonville,295,6,Sunday,4,9,7,32277,322,04,223.776708


In [31]:
ct = pd.crosstab(reg_df["day_of_week"], reg_df["parking status"])

chi2, p, dof, expected = chi2_contingency(ct)
print(chi2)
print(p)

n = ct.to_numpy().sum()
phi2 = chi2 / n
r, k = ct.shape
cramers_v = np.sqrt(phi2 / min(r - 1, k - 1))
print("Cramer's V:", cramers_v)

27.236242402430683
0.00714406860578781
Cramer's V: 0.1162903994412262


In [32]:
ct = pd.crosstab(reg_df["hour"], reg_df["parking status"])

chi2, p, dof, expected = chi2_contingency(ct)
print(chi2)
print(p)

n = ct.to_numpy().sum()
phi2 = chi2 / n
r, k = ct.shape
cramers_v = np.sqrt(phi2 / min(r - 1, k - 1))
print("Cramer's V:", cramers_v)

134.23146797088825
1.443704090156624e-10
Cramer's V: 0.2581650434467053


In [33]:
ct = pd.crosstab(reg_df["month"], reg_df["parking status"])

chi2, p, dof, expected = chi2_contingency(ct)
print(chi2)
print(p)

n = ct.to_numpy().sum()
phi2 = chi2 / n
r, k = ct.shape
cramers_v = np.sqrt(phi2 / min(r - 1, k - 1))
print("Cramer's V:", cramers_v)

106.67198587009663
4.3141797456650075e-13
Cramer's V: 0.23014177430679714


In [34]:
ct = pd.crosstab(reg_df["date"], reg_df["parking status"])

chi2, p, dof, expected = chi2_contingency(ct)
print(chi2)
print(p)

n = ct.to_numpy().sum()
phi2 = chi2 / n
r, k = ct.shape
cramers_v = np.sqrt(phi2 / min(r - 1, k - 1))
print("Cramer's V:", cramers_v)

89.95872587165857
0.007396031375304648
Cramer's V: 0.21134496933541047


In [35]:


# encode categoricals as integers
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(reg_df["parking status"])

X_enc = reg_df[["hour", "day_of_week", "month", "date", "avg_traffic", "pin id"]].copy()
for col in X_enc.columns:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col])

mi = mutual_info_classif(X_enc, y_encoded, discrete_features=True)
for col, val in zip(X_enc.columns, mi):
    print(col, "MI:", val)


hour MI: 0.07338228061470303
day_of_week MI: 0.013540452287632891
month MI: 0.05417054632546038
date MI: 0.04507947079939821
avg_traffic MI: 0.24297756253984026
pin id MI: 0.0


In [36]:
def eta_squared(df, numeric_col, categorical_col):
    groups = [df[df[categorical_col] == cat][numeric_col].dropna()
              for cat in df[categorical_col].unique()]

    # total sum of squares
    grand_mean = df[numeric_col].mean()
    ss_total = ((df[numeric_col] - grand_mean) ** 2).sum()

    # between-group sum of squares
    ss_between = sum([
        len(g) * (g.mean() - grand_mean) ** 2
        for g in groups
    ])

    eta_sq = ss_between / ss_total
    return eta_sq


# Run it
eta2 = eta_squared(reg_df, "avg_traffic", "parking status")
print("Eta-squared:", eta2)

Eta-squared: 0.04571536534282361
